In [56]:
import pandas as pd

In [57]:
data =pd.read_csv("twitter_training.csv", header=None)

In [58]:
data.head()

,0,1,2,3
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [59]:
data.drop([0,1], axis = 1, inplace=True)

In [60]:
data.head()

,2,3
0,Positive,im getting on borderlands and i will murder yo...
1,Positive,I am coming to the borders and I will kill you...
2,Positive,im getting on borderlands and i will kill you ...
3,Positive,im coming on borderlands and i will murder you...
4,Positive,im getting on borderlands 2 and i will murder ...


In [61]:
data.rename({2:"Sentiment", 3:"Review"}, axis = 1, inplace=True)

In [62]:
data.isna().sum()

Sentiment      0
Review       686
dtype: int64

In [63]:
data.dropna(inplace=True)

In [64]:
data.isna().sum()

Sentiment    0
Review       0
dtype: int64

In [65]:
data.duplicated().sum()

4227

In [66]:
data.drop_duplicates(inplace=True)

In [67]:
data

,Sentiment,Review
0,Positive,im getting on borderlands and i will murder yo...
1,Positive,I am coming to the borders and I will kill you...
2,Positive,im getting on borderlands and i will kill you ...
3,Positive,im coming on borderlands and i will murder you...
4,Positive,im getting on borderlands 2 and i will murder ...
...,...,...
74677,Positive,Just realized that the Windows partition of my...
74678,Positive,Just realized that my Mac window partition is ...
74679,Positive,Just realized the windows partition of my Mac ...
74680,Positive,Just realized between the windows partition of...


In [68]:
data.columns

Index(['Sentiment', 'Review'], dtype='object')

In [69]:
data['Sentiment'].value_counts()

Sentiment
Negative      21237
Positive      19138
Neutral       17110
Irrelevant    12284
Name: count, dtype: int64

# Preprocesssing

In [70]:
# lower or upper
# space 
# emoji
# punctuation 
# stopwords

In [71]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [72]:
import nltk  #  Natural language tool kit

In [73]:
from nltk.corpus import stopwords

In [74]:
import re  # regular exprsssion
def preprocess(text):
    text = text.lower()  # lower 
    text = re.sub("\s+", " ", text)  # extra space remove ho gaye
    text = "".join([a for a in text if a not in string.punctuation])
    text = " ".join([a for a in text.split()  if a not in stopwords.words("english")])   
    return text

<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Nasir Hussain\AppData\Local\Temp\ipykernel_22264\3786426270.py:4: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub("\s+", " ", text)  # extra space remove ho gaye


In [75]:
data["Review"]=data["Review"].apply(preprocess)

In [76]:
text = "this 😂 movie is bad"

In [77]:
text.encode("ascii", "ignore").decode("ascii")

'this  movie is bad'

In [78]:
data["Sentiment"].value_counts()

Sentiment
Negative      21237
Positive      19138
Neutral       17110
Irrelevant    12284
Name: count, dtype: int64

# Tokenization

In [79]:
data['Sentiment']=data['Sentiment'].map({
    "Negative"  :   0,
    "Positive"  :  1,
    "Neutral": 2,
    "Irrelevant" :3
    
})

In [80]:
data['Sentiment'].value_counts()

Sentiment
0    21237
1    19138
2    17110
3    12284
Name: count, dtype: int64

In [81]:
import tensorflow

In [82]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [83]:
tok = Tokenizer()

In [84]:
tok.fit_on_texts(data["Review"])

In [85]:
data["Review"]= tok.texts_to_sequences(data["Review"])

In [ ]:
tok.word_index

In [ ]:
data

In [88]:
max([len(a) for a in data["Review"]])

163

# Paddding 

In [89]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [91]:
X = pad_sequences(data["Review"], maxlen=163, padding="post")

In [92]:
X

array([[    5,    83,    59, ...,     0,     0,     0],
       [  283,  6367,   311, ...,     0,     0,     0],
       [    5,    83,    59, ...,     0,     0,     0],
       ...,
       [ 1771,   471, 12311, ...,     0,     0,     0],
       [ 1771,   471, 12311, ...,     0,     0,     0],
       [    2,   471, 12311, ...,     0,     0,     0]])

In [99]:
y=data["Sentiment"]
y.value_counts()

Sentiment
0    21237
1    19138
2    17110
3    12284
Name: count, dtype: int64

In [100]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.25, random_state=42)

In [101]:
from tensorflow.keras import layers
from tensorflow import keras

In [102]:
model = keras.Sequential([
    layers.SimpleRNN(128, activation="relu",return_sequences=False),
    layers.Dense(4, activation="softmax")
])

In [103]:
model.compile(optimizer="adam", loss = "sparse_categorical_crossentropy", metrics=["accuracy"])

In [ ]:
# model.fit(X_train, y_train, epochs=10, validation_data=0.20)

In [104]:
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# 1. Define vocab size (number of unique words + 1 for padding/OOV)
vocab_size = len(tok.word_index) + 1

# 2. Build the model with an Embedding layer
model = keras.Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=163),
    layers.SimpleRNN(128, activation="relu", return_sequences=False),
    layers.Dense(4, activation="softmax")
])

# 3. Compile the model
model.compile(
    optimizer="adam", 
    loss="sparse_categorical_crossentropy", 
    metrics=["accuracy"]
)


# Re-run train_test_split if needed to ensure y matches
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# 5. Fit the model using validation_data as a tuple
history = model.fit(
    X_train, 
    y_train, 
    epochs=10, 
    validation_data=(X_test, y_test)
)

Epoch 1/10


C:\Users\Nasir Hussain\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1636/1636 ━━━━━━━━━━━━━━━━━━━━ 87s 52ms/step - accuracy: 0.3004 - loss: 1.3697 - val_accuracy: 0.3035 - val_loss: 1.3708
Epoch 2/10
1636/1636 ━━━━━━━━━━━━━━━━━━━━ 82s 50ms/step - accuracy: 0.2984 - loss: 12.3458 - val_accuracy: 0.3035 - val_loss: 1.3705
Epoch 3/10
1636/1636 ━━━━━━━━━━━━━━━━━━━━ 87s 53ms/step - accuracy: 0.3061 - loss: 1.3662 - val_accuracy: 0.3035 - val_loss: 1.3714
Epoch 4/10
1636/1636 ━━━━━━━━━━━━━━━━━━━━ 89s 54ms/step - accuracy: 0.3016 - loss: 1.3676 - val_accuracy: 0.3035 - val_loss: 1.3712
Epoch 5/10
1636/1636 ━━━━━━━━━━━━━━━━━━━━ 89s 54ms/step - accuracy: 0.3001 - loss: 1.3677 - val_accuracy: 0.3035 - val_loss: 1.3724
Epoch 6/10
1636/1636 ━━━━━━━━━━━━━━━━━━━━ 87s 53ms/step - accuracy: 0.3045 - loss: 1.3670 - val_accuracy: 0.3035 - val_loss: 1.3698
Epoch 7/10
1636/1636 ━━━━━━━━━━━━━━━━━━━━ 88s 54ms/step - accuracy: 0.3079 - loss: 1.3670 - val_accuracy: 0.3035 - val_loss: 1.3707
Epoch 8/10
1636/1636 ━━━━━━━━━━━━━━━━━━━━ 88s 54ms/step - accuracy: 0.3030 - loss: 1.3